<a href="https://colab.research.google.com/github/bhavana-2005-stack/GENERATIVE_AI/blob/main/LangChain/LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-core
!pip install -q langchain-ollama
!pip install -q langchain-text-splitters
!pip install -q pypdf
!pip install -q sentense-transformers
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 20.3 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement sentense-transformers (from versions: none)
ERROR: No matching distribution found for sentense-transformers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.3 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama

#Load PDF

In [ ]:
loader=PyPDFLoader('employee_policy.pdf')
documents=loader.load()

print("Total pages",len(documents))
print(documents[0].page_content[:1000])

Total pages 1
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home Policy 
Employees may work from home twice per week. Manager approval is required for 
additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is 
available for spouse and children.


#Chunking

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=splitter.split_documents(documents)
print("Total Chunks:",len(chunks))
print(chunks[0].page_content)

Total Chunks: 2
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home Policy 
Employees may work from home twice per week. Manager approval is required for 
additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is


## Embedding

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

/tmp/ipykernel_2138/2352557002.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


## Create FAISS Vector Store

In [ ]:
vector_store = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS Create Successfully")

FAISS Create Successfully


## Create Retriever

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever Ready")

Retriever Ready


## Test Retrieval

In [ ]:
query = "How Many casual leaves are allowed?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n Chunk {i+1}")
    print("-" * 50)
    print(doc.page_content)


 Chunk 1
--------------------------------------------------
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home Policy 
Employees may work from home twice per week. Manager approval is required for 
additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is

 Chunk 2
--------------------------------------------------
employees are covered under company medical insurance. Dependent coverage is 
available for spouse and children.


## Connect Ollama

In [ ]:
llm = Ollama(model='llama3')

print("Connected to Ollama")

## Build Context

In [ ]:
context = "\n".join(
    [doc.page_content for doc in docs]
)

print(context)

Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick 
leaves annually. Unused casual leaves cannot be carried forward. Work From Home Policy 
Employees may work from home twice per week. Manager approval is required for 
additional remote work. Travel Policy Travel expenses are reimbursed within 30 days. 
Original receipts must be submitted for reimbursement. Medical Insurance Policy All 
employees are covered under company medical insurance. Dependent coverage is
employees are covered under company medical insurance. Dependent coverage is 
available for spouse and children.
